<a href="https://colab.research.google.com/github/mrunmayee3108/NeuroSolve/blob/main/gemma_baselineipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y transformers
!pip install -U transformers accelerate bitsandbytes sentencepiece

Found existing installation: transformers 5.10.2
Uninstalling transformers-5.10.2:
  Successfully uninstalled transformers-5.10.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd
import re

In [3]:
def extract_final_num(text):
    match = re.search(r"Final\s+Answer:\s*(-?\d+\.?\d*)", text, re.IGNORECASE)
    if match:
        return match.group(1)
    nums = re.findall(r'-?\d+\.?\d*', text)
    if nums:
        return nums[-1]
    return None

In [4]:
test_df = pd.read_csv("unified_svamp_test.csv").head(100)

In [5]:
import os
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-13.0/lib64'

In [6]:
from google.colab import userdata
from huggingface_hub import login

In [7]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token)
print("Authenticated successfully!")

Authenticated successfully!


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
model_id = "google/gemma-2b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
print("Downloading the model")
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
print("Sucess! The engine is running")

In [10]:
results = []
correct_count = 0
total_count = len(test_df)

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
print("\nStarting Baseline Evaluation-->")
for index, row in test_df.iterrows():
    question = row['question']
    ground_truth = str(row['answer']).strip()
    prompt = f"You are an expert math tutor. Question: {question}\nAnswer the question step-by-step. You must strictly end your response with 'Final Answer: <number>'.\nAnswer: "
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=True,
        repetition_penalty=1.15
    )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    extracted_answer = extract_final_num(response)

    is_correct = False
    if extracted_answer is not None:
        try:
            if float(extracted_answer) == float(ground_truth):
                is_correct = True
                correct_count += 1
        except ValueError:
            pass

    print(f"\n--- Problem {index + 1} ---")
    print(f"Question: {question}")
    print(f"Model Output:\n{response.strip()}")
    print(f"Extracted Answer: {extracted_answer}")
    print(f"Ground Truth Answer: {ground_truth}")
    print(f"Correct: {is_correct}")
    print("-" * 30)

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]


Starting Baseline Evaluation-->


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- Problem 1 ---
Question: There are 87 oranges and 290 bananas in Philip's collection. If the bananas are organized into 2 groups and oranges are organized into 93 groups How big is each group of bananas?
Model Output:
14

**Explanation:**

Number of bananas in 2 groups = 290 ÷ 2 = 145

Each group has 145 bananas.

Number of oranges in 93 groups = 87 ÷ 93 = 0.93

There are 0.93 groups of oranges. Each group has 10 oranges.

Therefore, each group of bananas has 14 and each group of oranges has 10.
Extracted Answer: 10.
Ground Truth Answer: 145
Correct: False
------------------------------

--- Problem 2 ---
Question: Marco and his dad went strawberry picking. Marco's dad's strawberries weighed 11 pounds. If together their strawberries weighed 30 pounds. How much did Marco's strawberries weigh?
Model Output:
29 pounds.

Step 1: Find the total weight of Marco's dad's strawberries.

11 pounds

Step 2: Subtract the total weight of Marco's dad's strawberries from the total weight of all t

In [13]:
accuracy = (correct_count / total_count) * 100
print(f"FINAL BASELINE ACCURACY: {accuracy:.2f}% ({correct_count}/{total_count})")

FINAL BASELINE ACCURACY: 25.00% (25/100)
